In [1]:
import torch
from torchvision import models
from torchvision.models import ResNet18_Weights, alexnet
import torchvision
from torch.utils.data import DataLoader
import os
from PIL import Image
from torch.utils.data import Dataset

class ImageNetValDataset(Dataset):
    def __init__(self, img_dir, label_file, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        
        # Load labels
        with open(label_file, "r") as f:
            self.labels = [int(line.strip().split()[1]) for line in f.readlines()]
        
        # Assume images are in alphabetical order (ILSVRC2012 convention: ILSVRC2012_val_00000001.JPEG ...)
        self.img_files = sorted(os.listdir(img_dir))
        assert len(self.img_files) == len(self.labels), "Mismatch between images and labels"

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_files[idx])
        img = Image.open(img_path).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            img = self.transform(img)

        return img, label

In [2]:
import torchvision


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
resnet18Model = torchvision.models.resnet18(pretrained=True).to(device)



Using device: cuda


c:\Users\jhon\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\jhon\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [3]:

weights = ResNet18_Weights.IMAGENET1K_V1

transform = weights.transforms()

# REPLACE, Path setup
val_img_dir = "./data/mp2/ILSVRC2012_img_val"          # REPLACE, directory with all 50k images
val_label_file = "./data/mp2/val.txt"  #REPLACE, file with 50k labels (one per line)
# Dataset + Loader


val_dataset = ImageNetValDataset(val_img_dir, val_label_file, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

In [4]:
import tqdm

correct_top1 = 0
correct_top5 = 0
total = 0

alexnet_top1 = 0
alexnet_top5 = 0

for images, labels in tqdm.tqdm(val_loader):
    # Convert to numpy array
    tensor_output = resnet18Model.forward(images.to(device))
    
    pred_top1 = tensor_output.argsort(dim=1, descending=True)[:, :1]  # Top-1 prediction
    pred_top5 = tensor_output.argsort(dim=1, descending=True)[:, :5]  # Top-5 predictions
    


    labels = labels.to(device)  # Move labels to the same device as images
    total += labels.size(0)
    correct_top1 += (pred_top1.squeeze() == labels).sum().item()
    correct_top5 += sum([labels[i].item() in pred_top5[i] for i in range(labels.size(0))])



100%|██████████| 782/782 [16:18<00:00,  1.25s/it]


In [ ]:
if total == 0:
    raise ValueError("Total number of samples is zero, cannot compute accuracy.")
acc1 = 100 * correct_top1 / total
acc5 = 100 * correct_top5 / total
print("Accuracy of ResNet-18 on Validation Set:")
print(f"Top-1 Accuracy: {acc1:.3f}%")
print(f"Top-5 Accuracy: {acc5:.3f}%")